# OLLAMA & LiteLLM et Introduction aux AgentIA

Prepare avec coeur par <a href="https://linkedin.com/in/dany-anderson-guimefack">Dany Anderson G.</a> et <a href="https://www.linkedin.com/in/ariste-yougbar%C3%A9-735387332/">Armel Ariste Y.</a> <br>
Email : dany.guimefack@centrale-casablanca.ma // wendenkomarmelariste.yougbare2@centrale-casablanca.ma

**Dans ce workshop**, nous allons :
1. Utiliser `Ollama` directement depuis Python.
2. Appeler l'API locale d'Ollama sur `http://localhost:11434`.
3. Exploiter `LiteLLM` comme interface unifiee pour communiquer avec differents modeles.
4. Introduire `Open-WebUI` comme interface et couche d'orchestration pour des agents locaux.

> Version exercice : certaines lignes sont volontairement incompletes. Completez-les en observant la structure des messages, des payloads HTTP et des reponses retournees.


---

## Partie 1  Ollama  : configuration et premier appel

## Qu'est-ce que Ollama ?

**Ollama** est un outil permettant d'exécuter des modèles d'intelligence artificielle (LLM) en local, directement sur votre machine, sans transfert de données vers le cloud.
On peut exécuter des modèles comme Llama, Mistral ou Gemma en local, via une API locale simple (`http://localhost:11434`).

## Pourquoi utiliser Ollama ?
Ollama est intéressant pour plusieurs raisons :
- exécuter des modèles localement.
- tester rapidement plusieurs modèles.
- éviter d'envoyer les données vers un service cloud.
- prototyper des applications IA.
- connecter facilement un modèle local à une API (FastAPI/Flask).
- travailler sans dépendre d'un compte OpenAI, Anthropic ou autre fournisseur.

In [ ]:
# Installation des bibliotheques necessaires

# Decommentez ces lignes si les paquets ne sont pas encore installes.
# %pip install ollama
# %pip install python-dotenv


### Commandes importantes - Ollama
Les commandes principales sont :
- `ollama --version` - verifier qu'Ollama est installe
- `ollama list` - voir les modeles installes
- `ollama pull <nom_du_modele>` - telecharger un modele
- `ollama run <nom_du_modele>` - lancer un modele en console
- `ollama ps` - voir les modeles en cours d'execution
- `ollama stop <nom_du_modele>` - arreter un modele
- `ollama rm <nom_du_modele>` - supprimer un modele

### Modele utilise dans la suite
Nous utiliserons `gemma3:1b` pour garder les appels rapides. Avant d'executer les cellules Python, verifiez que le modele est disponible avec :

```bash
ollama pull gemma3:1b
```


In [ ]:
# A executer dans un terminal si necessaire, pas dans Python :
# ollama pull gemma3:1b
# ollama run gemma3:1b


### Discuter avec le modele


In [ ]:
# Sans API locale : appel direct avec la bibliotheque ollama

from ollama import chat

MODEL_NAME = "gemma3:1b"

# TODO 1: formulez une question simple pour le modele.
question = ...

# TODO 2: construisez le message au format attendu par l'API chat.
# Indice: un message est un dictionnaire avec les cles "role" et "content".
messages = [
    {"role": ..., "content": ...}
]

conversation = chat(
    model=MODEL_NAME,
    messages=messages,
    stream=True,  # recevoir la reponse morceau par morceau
)

for morceau in conversation:
    contenu = morceau.message.content or ""
    print(contenu, end="", flush=True)


In [ ]:
# Avec l'API locale http://localhost:11434

import requests

OLLAMA_API_URL = "http://localhost:11434"
CHAT_ENDPOINT = "/..."  # TODO: remplacez par l'endpoint Ollama du chat


def converser_avec_modele(question: str) -> str:
    # TODO: completez le payload envoye a Ollama.
    # Indices: le modele est MODEL_NAME, stream=False, et messages contient la question utilisateur.
    payload = {
        "model": ...,
        "messages": [
            {"role": ..., "content": ...}
        ],
        "stream": ...,
    }

    reponse = requests.post(f"{OLLAMA_API_URL}{CHAT_ENDPOINT}", json=payload)
    # TODO: faites echouer la cellule clairement si Ollama renvoie une erreur HTTP.
    # Indice: les objets Response de requests ont une methode dediee.
    ...
    data = reponse.json()

    # TODO: extrayez uniquement le texte de l'assistant.
    # Indice: inspectez la structure JSON retournee par /api/chat.
    return ...


# TODO: appelez la fonction avec une question courte pour verifier le flux complet.
reponse_api = ...
print(reponse_api)


---

## Exemple Pratique : Questions sur un Article

Nous allons utiliser le **prompt système** pour fournir un article au modèle, puis créer une fonction qui répond aux questions en se basant **uniquement** sur cet article.

C'est une application concrète du pattern **RAG simplifié** (Retrieval-Augmented Generation) : on injecte un contexte dans le prompt système, et le modèle répond à partir de ce contexte uniquement.

In [ ]:
from ollama import chat

ARTICLE = """
Anthropic a conclu un partenariat strategique avec SpaceX afin d'utiliser l'ensemble de la capacite
informatique du centre de donnees Colossus 1, situe a Memphis dans le Tennessee. Grace a cet accord,
l'entreprise specialisee dans l'intelligence artificielle beneficiera de plus de 300 megawatts de
puissance de calcul. Le groupe a egalement indique envisager une collaboration future avec SpaceX
autour du developpement d'infrastructures de plusieurs gigawatts dans l'espace.

L'accord doit permettre a Anthropic d'ameliorer les performances proposees aux abonnes de ses offres
Claude Pro et Claude Max. Cette alliance intervient pourtant dans un contexte de tensions publiques
entre Anthropic et Elon Musk, proprietaire de SpaceX et fondateur de xAI. Ces derniers mois, le
milliardaire a multiplie les critiques contre Anthropic, notamment sur les questions de regulation de
l'intelligence artificielle, allant jusqu'a accuser l'entreprise de \"detester la civilisation occidentale\".

Parallelement, xAI poursuit l'expansion rapide de ses infrastructures a Memphis afin de concurrencer
OpenAI, Google et Anthropic sur le marche de l'IA. Le centre Colossus 1 est actuellement son plus
important site de calcul. Le projet fait toutefois face a des critiques locales liees a l'utilisation
de turbines au gaz naturel, accusees d'aggraver la pollution atmospherique dans la region. Par ailleurs,
la startup fondee en 2021 par d'anciens chercheurs et dirigeants d'OpenAI serait en discussion avec
des investisseurs pour une levee de fonds susceptible de valoriser l'entreprise jusqu'a 900 milliards
de dollars.
"""


def repond_a_partir_de_larticle(question: str) -> str:
    """Repond a une question en se basant uniquement sur l'article fourni."""
    consigne_systeme = (
        "Tu reponds uniquement avec les informations de l'article. "
        "Si la reponse ne figure pas dans l'article, dis-le clairement.\n\n"
        f"Article :\n{ARTICLE}"
    )

    # TODO: construisez les deux messages: systeme puis utilisateur.
    # Indice: le premier injecte l'article, le second contient la question recue en parametre.
    messages = [
        {"role": ..., "content": ...},
        {"role": ..., "content": ...},
    ]

    conversation = chat(model=MODEL_NAME, messages=messages, stream=True)
    morceaux = []

    for morceau in conversation:
        contenu = morceau.message.content or ""
        print(contenu, end="", flush=True)
        # TODO: gardez aussi une copie du texte pour pouvoir le retourner a la fin.
        ...

    # TODO: retournez la reponse complete sous forme d'une seule chaine.
    return ...


In [ ]:
# Test de la fonction avec plusieurs questions
questions = [
    "Quel est l'accord conclu entre Anthropic et SpaceX ?",
    # TODO: ajoutez une question dont la reponse est explicitement dans l'article.
    ...,
    # TODO: ajoutez une question hors article pour verifier que le modele refuse d'inventer.
    ...,
]

for question in questions:
    print(f"Question : {question}")
    print("Reponse : ", end="")
    repond_a_partir_de_larticle(question)
    print("\n" + "-" * 70)


---

## Partie 2 LiteLLM

Qu'est-ce que LiteLLM ?

`LiteLLM` est une couche d'abstraction qui unifie les appels à différents fournisseurs et moteurs de modèles sous une même API.

Sans `LiteLLM`, chaque fournisseur a sa propre syntaxe d'appel ; avec `LiteLLM`, on garde un format d'appel commun, ce qui facilite le changement de modèle et l'intégration.

Pour la documentation, voir : https://docs.litellm.ai/docs/

Avantages :
- Même format d'appel pour plusieurs providers.
- Facilité pour changer de modèle sans réécrire la logique d'appel.

In [ ]:
#Installation de LiteLLM
%pip install litellm==1.83.1

In [ ]:
# Exemple avec LiteLLM: meme format de messages, autre couche d'appel

from litellm import completion

# TODO: completez le nom du modele LiteLLM pour cibler Ollama.
# Indice: LiteLLM utilise le prefixe "ollama/" avant le nom du modele.
modele_litellm = ...

# TODO: reutilisez le format role/content pour demander une reponse courte.
messages_litellm = [
    {"role": ..., "content": ...}
]

response = completion(
    model=modele_litellm,
    messages=messages_litellm,
    # TODO: indiquez l'adresse de l'API Ollama locale.
    api_base=...,
)

# TODO: affichez le contenu du premier choix retourne par LiteLLM.
print(...)


---

## Partie 3 OpenWebUI pour le système agentique 

But : intégrer `Open-WebUI` comme couche d'orchestration et d'interface pour un système agentique local.

Rôle principal :
- Fournir une interface web pour interagir avec les modèles locaux et gérer les plugins/outils nécessaires aux agents.
- Exposer des endpoints (REST/WebSocket) permettant à des agents programmatiques d'appeler des modèles et des plugins.

Capacités utiles pour un agent local :
- Hébergement et commutation de modèles locaux (permet de tester plusieurs backends).
- Gestion des conversations et persistance des sessions (historique utile pour l'état de l'agent).
- Intégration de plugins/outils (recherche web, exécution de code, récupérateurs de documents).
- Streaming des réponses pour réduire la latence perçue.

Flux de travail (comment cela s'articule avec Ollama et LiteLLM) :
1. Déployer `Open-WebUI` (Docker) pour disposer d'une UI et d'un ensemble de plugins prêts à l'emploi.
2. Utiliser `Ollama` pour télécharger et exécuter les modèles localement (ils deviennent accessibles via API locales).
3. Appeler les modèles depuis vos scripts/agents via `LiteLLM` : `LiteLLM` peut cibler `Ollama` ou l'API d'Open-WebUI selon la configuration.
4. Lorsqu'un agent doit utiliser un outil (ex. recherche ou exécution de code), déclencher le plugin correspondant via Open-WebUI.
5. Récupérer le résultat de l'outil, le fournir au modèle via `LiteLLM`, et poursuivre l'itération de raisonnement jusqu'à l'objectif.

Notes pratiques :
- Conserver `Ollama` pour le service des modèles, `LiteLLM` pour l'orchestration des appels, et `Open-WebUI` pour la supervision, les plugins et l'interface.
- Open-WebUI facilite l'ajout de capacités externes (outils) tout en conservant l'exécution locale des modèles.

### Utiliser Ollama sur un réseau local avec ngrok

Oui, c'est possible. L'idée est de créer un tunnel sécurisé vers le port `11434` d'Ollama, puis de donner à OpenWebUI ou à LiteLLM l'URL publique générée.

Étapes :
1. Lancer Ollama en local et vérifier qu'il répond sur `http://localhost:11434`.
2. Installer et connecter `ngrok` à votre compte.
3. Ouvrir un tunnel vers Ollama : `ngrok http 11434`.
4. Récupérer l'URL HTTPS fournie par ngrok.
5. Configurer OpenWebUI ou LiteLLM avec cette URL comme base distante au lieu de `localhost`.
6. Partager l'URL à votre ami pour qu'il puisse accéder à votre instance depuis son appareil.

Points d'attention :
- Exposer directement le port d'Ollama sur Internet demande des précautions.
- Ajoutez un contrôle d'accès si possible et évitez d'ouvrir le tunnel plus longtemps que nécessaire.
- Si votre ami est sur le même Wi-Fi, un simple accès réseau local peut suffire ; ngrok est surtout utile quand vous voulez traverser le réseau local sans configuration supplémentaire.

---
